#   E. coli coding analysis template

##  Description

This notebook is a starting point for analysis of the *E. coli* genome. It reuses the RefSeq loading workflow and binary-signal construction logic from the EDA notebook.

In [1]:
import io
import math
from enum import Enum

import numpy as np
import pandas as pd
import pywt
from Bio import Entrez, SeqIO
import Bio

ACCESSION = "NC_000913.3"  # E. coli K-12 MG1655 complete genome
EMAIL = "daniel_mcgonigle1@student.uml.edu"  # NCBI asks for an email

Entrez.email = EMAIL


In [2]:
def fetch_genbank_record(accession: str) -> Bio.SeqRecord.SeqRecord:
    """
    Fetch a nucleotide sequence record from NCBI and parse it into a Biopython SeqRecord.

    Parameters
    ----------
    accession : str
        RefSeq or GenBank accession for the genome of interest.

    Returns
    -------
    Bio.SeqRecord.SeqRecord
        Parsed record containing the DNA sequence and feature annotations.
    """
    with Entrez.efetch(db="nuccore", id=accession, rettype="gbwithparts", retmode="text") as handle:
        text = handle.read()
    return SeqIO.read(io.StringIO(text), "genbank")


class SignalType(str, Enum):
    """Signal domain choices for genome-derived indicator signals."""

    FULL = "full"
    CDS = "cds"
    NONCDS = "noncds"


record = fetch_genbank_record(ACCESSION)
sequence = str(record.seq).upper()
genome_len = len(sequence)

print(f"Record ID: {record.id}")
print(f"Description: {record.description}")
print(f"Genome length: {genome_len:,} bp")
print(f"Annotated feature types: {sorted({feat.type for feat in record.features})}")


Record ID: NC_000913.3
Description: Escherichia coli str. K-12 substr. MG1655, complete genome
Genome length: 4,641,652 bp
Annotated feature types: ['CDS', 'gene', 'misc_feature', 'mobile_element', 'ncRNA', 'rRNA', 'rep_origin', 'source', 'tRNA']


In [3]:
def build_cds_mask(record: Bio.SeqRecord.SeqRecord, sequence_length: int) -> np.ndarray:
    """
    Build a boolean mask marking every base position covered by a CDS annotation.

    Parameters
    ----------
    record : Bio.SeqRecord.SeqRecord
        Record whose CDS annotations define coding positions.
    sequence_length : int
        Length of the full genome sequence.

    Returns
    -------
    np.ndarray
        Boolean mask whose True entries indicate genome positions covered by at
        least one CDS interval.
    """
    cds_mask = np.zeros(sequence_length, dtype=bool)
    for feat in record.features:
        if feat.type != "CDS":
            continue

        parts = getattr(feat.location, "parts", [feat.location])
        for part in parts:
            start = int(part.start)
            end = int(part.end)
            cds_mask[start:end] = True

    return cds_mask


def extract_signal_sequence(
    sequence: str,
    record: Bio.SeqRecord.SeqRecord,
    signal_type: SignalType,
) -> str:
    """
    Extract the sequence corresponding to a requested signal domain.

    Parameters
    ----------
    sequence : str
        Full nucleotide sequence.
    record : Bio.SeqRecord.SeqRecord
        Record whose CDS annotations define coding and non-coding positions.
    signal_type : SignalType
        Which sequence domain to extract: full genome, CDS-only, or the
        complement of CDS.

    Returns
    -------
    str
        Sequence associated with the selected signal type.

    Notes
    -----
    SignalType.CDS is built by concatenating CDS feature intervals in record order.
    SignalType.NONCDS is built as the complement of the CDS mask across the full genome,
    so it cannot exceed the full-genome length.
    """
    if signal_type == SignalType.FULL:
        return sequence

    cds_mask = build_cds_mask(record, len(sequence))

    if signal_type == SignalType.CDS:
        return "".join(np.asarray(list(sequence))[cds_mask])
    if signal_type == SignalType.NONCDS:
        return "".join(np.asarray(list(sequence))[~cds_mask])

    raise ValueError(f"Unsupported signal_type: {signal_type}")


def create_binary_signal(
    bases: str,
    sequence: str,
    record: Bio.SeqRecord.SeqRecord,
    signal_type: SignalType = SignalType.FULL,
) -> np.ndarray:
    """
    Create a binary nucleotide indicator signal.

    Parameters
    ----------
    bases : str
        One or more nucleotide letters to mark with 1s, for example "GC" or "A".
    sequence : str
        Full sequence from which the signal is built.
    record : Bio.SeqRecord.SeqRecord
        Record whose CDS annotations determine the selected signal domain.
    signal_type : SignalType, default SignalType.FULL
        Which domain to encode: full genome, CDS-only, or non-CDS as the
        complement of CDS.

    Returns
    -------
    np.ndarray
        One-dimensional int8 array of 0s and 1s.
    """
    selected_bases = {base.upper() for base in bases if not base.isspace()}
    if not selected_bases:
        raise ValueError("bases must contain at least one nucleotide letter")

    sequence_to_encode = extract_signal_sequence(sequence, record, signal_type)

    return np.fromiter(
        (1 if base in selected_bases else 0 for base in sequence_to_encode),
        dtype=np.int8,
    )


In [7]:
gc_signal = create_binary_signal("GC", sequence, record=record, signal_type=SignalType.FULL)
gc_cds_signal = create_binary_signal("GC", sequence, record=record, signal_type=SignalType.CDS)
gc_noncds_signal = create_binary_signal("GC", sequence, record=record, signal_type=SignalType.NONCDS)

print("Full GC signal length:", len(gc_signal))
print("CDS GC signal length:", len(gc_cds_signal))
print("Non-CDS GC signal length:", len(gc_noncds_signal))
print("Full GC fraction:", gc_signal.mean())
print("CDS GC fraction:", gc_cds_signal.mean())
print("Non-CDS GC fraction:", gc_noncds_signal.mean())


Full GC signal length: 4641652
CDS GC signal length: 4003900
Non-CDS GC signal length: 637752
Full GC fraction: 0.5079070985933456
CDS GC fraction: 0.5191708084617498
Non-CDS GC fraction: 0.43719188650133595
